# Amazon ML Challenge 2026 — Business Entity Resolution

v2 pipeline: normalize → key index (cap 1200/600) → learned pre-ranker pruning → noise-model + string features → LightGBM → one-to-one selection.
Settings: **Accelerator: None (CPU)**, **Internet: On**. Add the challenge data as an Input.
Run with **Save Version → Save & Run All (Commit)** — only committed versions keep their output files.
Outputs kept for the team: `output/*.tsv`, `work/model.pkl`, `work/prerank.txt`, `work/holdout_preds.parquet`,
`work/test_scores.parquet`, `work/*_cands.parquet`, `work/*_norm_*.parquet`, `work/train_report.json`, `run.log`.

In [ ]:
# Offline-safe install: use wheels from an attached dataset if present, else PyPI (needs Internet On)
import glob, os, sys, subprocess
whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
try:
    import unidecode, rapidfuzz
    print("already installed")
except ImportError:
    if whls:
        dirs = sorted({os.path.dirname(w) for w in whls})
        cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-index"] + sum([["--find-links", d] for d in dirs], []) + ["unidecode", "rapidfuzz"]
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q", "unidecode==1.3.8", "rapidfuzz==3.9.7"]
    print(subprocess.run(cmd, capture_output=True, text=True).stderr[-2000:])
import unidecode, rapidfuzz, lightgbm, pandas, numpy
print("python", sys.version.split()[0], "rapidfuzz", rapidfuzz.__version__, "lightgbm", lightgbm.__version__, "pandas", pandas.__version__, "numpy", numpy.__version__)
!nproc; free -g

In [ ]:
os.makedirs("/kaggle/working/ber/src", exist_ok=True)
%cd /kaggle/working/ber

In [ ]:
%%writefile src/__init__.py
# business entity resolution pipeline

In [ ]:
%%writefile src/blocking.py
"""Candidate generation: IDF-weighted multi-key token blocking, fully vectorized.

Every record emits a set of hashed keys (country-scoped):
  name  : phonetic-skeleton tokens, 5-char prefixes, skeleton token pairs
  addr  : skeleton tokens, (house-number, token) pairs
An S1 record and a pool record share a key -> they accumulate idf(key) into a name or
address score. Keys that are too common (pool df > cap) are dropped. For every S1 we
keep the union of the top-K by total / name / address score. All joins are numpy
range-expansions over sorted key arrays (no Python loop per pair).
"""
import os
import multiprocessing as mp
import numpy as np
import pandas as pd

from .normalize import skeleton

STREET_TYPES = {"st", "ave", "rd", "dr", "ln", "cir", "ct", "blvd", "pl", "pkwy", "hwy", "ter", "trl",
                "sq", "cv", "way", "loop", "n", "s", "e", "w", "ne", "nw", "se", "sw", "ste", "apt", "flr",
                "bldg", "sec", "ngr", "col", "marg", "rue", "all", "chem", "imp", "rte", "fbg", "quai",
                "crs", "res", "main", "new", "old"}
MAX_PAIR_TOKS = 6


MAX_CROSS_NAME, MAX_CROSS_ADDR, MAX_ADDR_PAIR = 4, 6, 6
EXTRA_KEYS = True
SINGLE_KINDS = set("trpja")


def record_keys(country, n_clean, a_clean, nums):
    """Returns (name_keys, addr_keys, cross_keys) as sets of strings."""
    c = country
    nk, ak, xk = set(), set(), set()
    ntoks = n_clean.split()
    sk = []
    for t in ntoks:
        s = skeleton(t)
        if len(s) >= 2:
            nk.add(f"t{c}|{s}")
            sk.append(s)
        elif len(t) >= 2:
            nk.add(f"r{c}|{t}")
        if len(t) >= 5:
            nk.add(f"p{c}|{t[:5]}")
    joined = "".join(ntoks)
    if len(joined) >= 8:                       # 'southernexports.com' <-> 'Southern Exports'
        nk.add(f"j{c}|{joined[:8]}")
    sk = list(dict.fromkeys(sk))[:MAX_PAIR_TOKS]
    if EXTRA_KEYS:
        # typo-tolerant name keys: pairs of 4-char prefixes ('consulatscy' ~ 'consultancy')
        pf = list(dict.fromkeys(t[:4] for t in ntoks if len(t) >= 3))[:MAX_PAIR_TOKS]
        for i in range(len(pf)):
            for j in range(i + 1, len(pf)):
                a, b = (pf[i], pf[j]) if pf[i] < pf[j] else (pf[j], pf[i])
                nk.add(f"c{c}|{a}|{b}")
    for i in range(len(sk)):
        for j in range(i + 1, len(sk)):
            a, b = (sk[i], sk[j]) if sk[i] < sk[j] else (sk[j], sk[i])
            nk.add(f"b{c}|{a}|{b}")
    if a_clean or nums:
        ask = []
        for t in a_clean.split():
            if t in STREET_TYPES:
                continue
            s = skeleton(t) if t.isalpha() else t
            if len(s) >= 2:
                ask.append(s)
        ask = list(dict.fromkeys(ask))
        for s in ask:
            ak.add(f"a{c}|{s}")
        nl = nums.split()[:2]
        for n in nl:
            for s in ask:
                ak.add(f"n{c}|{n}|{s}")
        ap = ask[:MAX_ADDR_PAIR]
        for i in range(len(ap)):
            for j in range(i + 1, len(ap)):
                a, b = (ap[i], ap[j]) if ap[i] < ap[j] else (ap[j], ap[i])
                ak.add(f"q{c}|{a}|{b}")
        for s_ in sk[:MAX_CROSS_NAME]:
            for a in ask[:MAX_CROSS_ADDR] + nl:
                xk.add(f"x{c}|{s_}|{a}")
        if EXTRA_KEYS:
            nu = list(dict.fromkeys(nums.split()))[:4]
            for i in range(len(nu)):          # number pairs: '5534 10480' survives a replaced name
                for j in range(i + 1, len(nu)):
                    a, b = (nu[i], nu[j]) if nu[i] < nu[j] else (nu[j], nu[i])
                    ak.add(f"u{c}|{a}|{b}")
    return nk, ak, xk


def _keys_chunk(args):
    off, countries, names, addrs, nums = args
    recs, keys, fams = [], [], []
    for i, (c, n, a, u) in enumerate(zip(countries, names, addrs, nums)):
        for fam, ks in enumerate(record_keys(c.lower(), n, a, u)):
            for k in ks:
                recs.append(off + i); keys.append(k)
                # fam code: +0 name / +1 addr / +2 cross ; +4 if single-token key (t,r,p,j,a)
                fams.append(fam + (4 if k[0] in SINGLE_KINDS else 0))
    h = pd.util.hash_array(np.array(keys, dtype=object), categorize=False)
    return np.array(recs, np.int32), h, np.array(fams, np.int8)


def build_keys(df, n_jobs=None, chunk=100_000):
    n_jobs = n_jobs or os.cpu_count()
    cols = [df.country.tolist(), df.n_clean.tolist(), df.a_clean.tolist(), df.nums.tolist()]
    parts = [(i, *[c[i:i + chunk] for c in cols]) for i in range(0, len(df), chunk)]
    if n_jobs > 1 and len(parts) > 1:
        with mp.get_context("fork").Pool(n_jobs) as pool:
            res = pool.map(_keys_chunk, parts)
    else:
        res = [_keys_chunk(p) for p in parts]
    return (np.concatenate([r[0] for r in res]), np.concatenate([r[1] for r in res]),
            np.concatenate([r[2] for r in res]))


def _group_rank(g, score):
    """rank (0 = best) of each row within its group g, by descending score.
    Single int64 argsort on (group, -score) instead of lexsort (~5x faster)."""
    v = np.asarray(score, np.float32)
    bits = v.view(np.int32).astype(np.int64)
    bits = np.where(bits < 0, np.int64(0x7FFFFFFF) - bits, bits)   # order-preserving for negatives
    key = (np.asarray(g, np.int64) << 32) | (np.int64(0xFFFFFFFF) - (bits + np.int64(0x80000000)) & np.int64(0xFFFFFFFF))
    order = np.argsort(key, kind="stable")
    gs = g[order]
    first = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    rank_sorted = np.arange(len(g)) - np.repeat(first, np.diff(np.r_[first, len(g)]))
    rank = np.empty(len(g), np.int32)
    rank[order] = rank_sorted
    return rank


class KeyIndex:
    """Sorted pool-key index + S1 key lists; build once, run blocking passes many times."""

    def __init__(self, s1, pool, cap=600, df_scale=1.0, n_jobs=None, verbose=True, cap_composite=None):
        import time
        t0 = time.time()
        pr, pk, pf = build_keys(pool, n_jobs)
        sr, sk, sf = build_keys(s1, n_jobs)
        if verbose:
            print(f"[block] keys pool={len(pk):,} s1={len(sk):,} ({time.time() - t0:.0f}s)", flush=True)
        order = np.argsort(pk)
        pk = pk[order]
        self.pr = pr[order]
        del order, pf, pr
        newk = np.empty(len(pk), bool)
        newk[0] = True
        np.not_equal(pk[1:], pk[:-1], out=newk[1:])
        self.starts = np.flatnonzero(newk)
        uk = pk[self.starts]
        del pk, newk
        self.counts = np.diff(np.r_[self.starts, len(self.pr)]).astype(np.int64)
        self.idf = np.log((len(pool) + 1) / (self.counts + 1)).astype(np.float32)
        pos = np.searchsorted(uk, sk)
        pos[pos == len(uk)] = 0
        capv = np.where(sf >= 4, cap, cap_composite or cap)
        ok = (uk[pos] == sk) & (self.counts[pos] * df_scale <= capv)
        self.sr, self.sf, self.pos = sr[ok], (sf[ok] & 3).astype(np.int8), pos[ok]
        self.n_s1 = len(s1)
        del sk, ok, uk
        if verbose:
            print(f"[block] cap={cap} usable s1 keys={len(self.sr):,} "
                  f"expansion={self.counts[self.pos].sum():,} ({time.time() - t0:.0f}s)", flush=True)


def generate_candidates(s1, pool, cap=600, df_scale=1.0, k_total=30, k_name=10, k_addr=10, k_cross=0,
                        budget=60_000_000, n_jobs=None, verbose=True, prerank=None, s1_subset=None, index=None):
    """Returns DataFrame [s1_idx, pool_idx, bs_*] (row indices into s1/pool).
    index: a prebuilt KeyIndex (reuse across passes). s1_subset: bool mask of S1 rows to process.
    prerank: callable(DataFrame)->DataFrame applied to every chunk's kept pairs (learned pruning)."""
    import time
    t0 = time.time()
    ix = index or KeyIndex(s1, pool, cap, df_scale, n_jobs, verbose)
    pr, starts, counts, idf = ix.pr, ix.starts, ix.counts, ix.idf
    sr, sf, pos = ix.sr, ix.sf, ix.pos
    if s1_subset is not None:
        m = s1_subset[sr]
        sr, sf, pos = sr[m], sf[m], pos[m]

    out = []
    # chunk S1 records so each chunk expands to <= budget pairs (bounded memory)
    rec_exp = np.bincount(sr, weights=counts[pos], minlength=len(s1))
    cum = np.cumsum(rec_exp)
    rb = np.unique(np.r_[0, np.searchsorted(cum, np.arange(budget, cum[-1] + budget, budget)), len(s1)])
    bounds = np.searchsorted(sr, rb)
    for b0, b1 in zip(bounds[:-1], bounds[1:]):
        if b1 <= b0:
            continue
        r, f, p = sr[b0:b1], sf[b0:b1], pos[b0:b1]
        cnt = counts[p]
        tot = int(cnt.sum())
        rep_idx = np.repeat(np.arange(len(r)), cnt)
        offs = np.arange(tot) - np.repeat(np.cumsum(cnt) - cnt, cnt) + np.repeat(starts[p], cnt)
        pairs = (r[rep_idx].astype(np.int64) << 24) | pr[offs].astype(np.int64)
        w = idf[p][rep_idx]
        fam = f[rep_idx]
        del offs, rep_idx
        up, inv = np.unique(pairs, return_inverse=True)
        del pairs
        s_name = np.bincount(inv, weights=w * (fam == 0), minlength=len(up)).astype(np.float32)
        s_addr = np.bincount(inv, weights=w * (fam == 1), minlength=len(up)).astype(np.float32)
        s_cross = np.bincount(inv, weights=w * (fam == 2), minlength=len(up)).astype(np.float32)
        n_nk = np.bincount(inv, weights=(fam == 0), minlength=len(up)).astype(np.float32)
        n_ak = np.bincount(inv, weights=(fam == 1), minlength=len(up)).astype(np.float32)
        n_xk = np.bincount(inv, weights=(fam == 2), minlength=len(up)).astype(np.float32)
        nkeys = np.bincount(inv, minlength=len(up)).astype(np.int16)
        g = (up >> 24).astype(np.int32)
        s_tot = s_name + s_addr + s_cross
        keep = (_group_rank(g, s_tot) < k_total) | (_group_rank(g, s_name) < k_name) | \
               (_group_rank(g, s_addr) < k_addr) | (_group_rank(g, s_cross) < k_cross)
        blk = pd.DataFrame({"s1_idx": g[keep], "pool_idx": (up[keep] & ((1 << 24) - 1)).astype(np.int32),
                            "bs_name": s_name[keep], "bs_addr": s_addr[keep], "bs_cross": s_cross[keep],
                            "bs_nkeys": nkeys[keep], "bs_nnk": n_nk[keep], "bs_nak": n_ak[keep],
                            "bs_nxk": n_xk[keep]})
        if prerank is not None:
            blk = prerank(blk)
        out.append(blk)
        del inv, w, fam, s_name, s_addr, s_cross, n_nk, n_ak, n_xk, nkeys, g, s_tot
        if verbose:
            print(f"[block] s1 rows {r[0]:,}-{r[-1]:,}: expanded {tot:,} -> uniq {len(up):,} -> kept {len(out[-1]):,}"
                  f" ({time.time() - t0:.0f}s)", flush=True)
    return pd.concat(out, ignore_index=True)

In [ ]:
%%writefile src/features.py
"""Pairwise features for (S1, pool) candidate pairs.

* String similarities: rapidfuzz.process.cpdist (C++, multithreaded, element-wise pairs).
* Set / noise-model / number features: lean Python in forked workers over chunks.
* Competition context (ranks / margins among candidates): vectorized numpy/pandas.
"""
import math
import os
import multiprocessing as mp
from collections import Counter

import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler, Levenshtein
from rapidfuzz.process import cpdist

from .noise import token_noise_feats, number_feats, TOKEN_COLS, NUM_COLS

_G = {}


# ------------------------------------------------------------------ IDF tables
def build_idf(series_list):
    c = Counter()
    n = 0
    for s in series_list:
        for v in s:
            n += 1
            c.update(set(v.split()))
    return {k: math.log((n + 1) / (v + 1)) for k, v in c.items()}, math.log(n + 1)


# ------------------------------------------------------------------ vectorized string sims
CPD = [  # (name, column, scorer, joined-without-spaces)
    ("n_ratio", "n_clean", fuzz.ratio, False),
    ("n_tset", "n_clean", fuzz.token_set_ratio, False),
    ("n_tsort", "n_clean", fuzz.token_sort_ratio, False),
    ("n_partial", "n_clean", fuzz.partial_ratio, False),
    ("n_jw_join", "n_clean", JaroWinkler.normalized_similarity, True),
    ("n_ratio_join", "n_clean", fuzz.ratio, True),
    ("n_partial_join", "n_clean", fuzz.partial_ratio, True),
    ("n_skel_tset", "nsk", fuzz.token_set_ratio, False),
    ("a_ratio", "a_clean", fuzz.ratio, False),
    ("a_tset", "a_clean", fuzz.token_set_ratio, False),
    ("a_partial", "a_clean", fuzz.partial_ratio, False),
    ("a_skel_tset", "ask", fuzz.token_set_ratio, False),
    ("u_tset", "nums", fuzz.token_set_ratio, False),
]


def cpdist_features(A, B, workers=-1):
    out = {}
    cache = {}
    for name, col, scorer, join in CPD:
        k = (col, join)
        if k not in cache:
            a, b = A[col], B[col]
            if join:
                cache[k] = ([x.replace(" ", "") for x in a], [x.replace(" ", "") for x in b])
            else:
                cache[k] = (list(a), list(b))
        a, b = cache[k]
        v = cpdist(a, b, scorer=scorer, workers=workers, dtype=np.float32)
        empty = np.fromiter((not x or not y for x, y in zip(a, b)), bool, len(a))
        v[empty] = -1
        out[name] = v
    return out


# ------------------------------------------------------------------ python per-pair features
def _wjacc(sa, sb, idf, dflt):
    if not sa or not sb:
        return -1.0, -1.0, 0.0
    inter = sa & sb
    wi = sum(idf.get(t, dflt) for t in inter)
    wu = wi + sum(idf.get(t, dflt) for t in sa ^ sb)
    miss = max((idf.get(t, dflt) for t in sa - sb), default=0.0)
    return wi / wu if wu else 0.0, len(inter) / len(sa | sb), miss


def _num_old(n1, n2):
    a, b = n1.split(), n2.split()
    if not a or not b:
        return -1.0, -1.0, -1.0, -1.0
    sa, sb = set(a), set(b)
    inter = len(sa & sb)
    near = 0.0
    if not inter:
        near = float(any(Levenshtein.distance(x, y) <= 1 for x in sa for y in sb if len(x) >= 3 or len(y) >= 3))
    return inter / len(sa | sb), float(a[0] == b[0]), float(inter == 0), near


PY_SRC = ["nsk", "ask", "legal", "nums", "state", "rn", "ra", "ru", "country"]
PY_COLS = (["n_wjacc", "n_jacc", "n_miss_idf", "n_miss_idf_r", "n_first_eq", "n_ntok1", "n_ntok2",
            "legal_eq", "legal_ov", "a_wjacc", "a_jacc", "a_miss_idf", "a_miss_idf_r",
            "num_jacc", "num_first_eq", "num_conflict", "num_near", "state_eq"]
           + ["nn_" + c for c in TOKEN_COLS] + ["an_" + c for c in TOKEN_COLS] + NUM_COLS)


def _py_row(k1, a1, l1, u1, st1, rn1, ra1, ru1, ct1, k2, a2, l2, u2, st2, rn2, ra2, ru2, ct2):
    idf_n, dn, idf_a, da, nadd, ndrop, aadd, adrop, fq_n, fq_a = _G["t"]
    t1, t2 = k1.split(), k2.split()
    s1, s2 = set(t1), set(t2)
    wj, jac, miss = _wjacc(s1, s2, idf_n, dn)
    _, _, miss_r = _wjacc(s2, s1, idf_n, dn)
    x1, x2 = set(a1.split()), set(a2.split())
    if x1 and x2:
        awj, ajac, amiss = _wjacc(x1, x2, idf_a, da)
        _, _, amiss_r = _wjacc(x2, x1, idf_a, da)
    else:
        awj = ajac = amiss = amiss_r = -1.0
    return ((wj, jac, miss, miss_r, float(t1[:1] == t2[:1]), float(len(t1)), float(len(t2)),
             -1.0 if (not l1 or not l2) else float(l1 == l2),
             -1.0 if (not l1 or not l2) else float(bool(set(l1.split()) & set(l2.split()))),
             awj, ajac, amiss, amiss_r)
            + _num_old(u1, u2)
            + (-1.0 if (not st1 or not st2) else float(st1 == st2),)
            + token_noise_feats(rn1.split(), rn2.split(), nadd, ndrop, ct1.lower(), fq_n)
            + token_noise_feats(ra1.split(), ra2.split(), aadd, adrop, ct1.lower(), fq_a)
            + number_feats(tuple(ru1.split()), tuple(ru2.split())))


def _py_chunk(cols):
    return np.array([_py_row(*r) for r in zip(*cols)], dtype=np.float32)


def _init(tables):
    _G["t"] = tables


def string_features(cands, s1, pool, tables, n_jobs=None, chunk=1_000_000, sub=100_000):
    """tables = (idf_n, dn, idf_a, da, name_add_lo, name_drop_lo, addr_add_lo, addr_drop_lo,
                  frequent_name_tokens, frequent_addr_tokens).
    Returns a float32 DataFrame (one preallocated block, no intermediate copies)."""
    n_jobs = n_jobs or os.cpu_count()
    ii, jj = cands.s1_idx.values, cands.pool_idx.values
    need = sorted({c for _, c, _, _ in CPD} | set(PY_SRC))
    S1 = {c: s1[c].array for c in need}
    PL = {c: pool[c].array for c in need}
    cols = [c for c, *_ in CPD] + PY_COLS
    out = np.empty((len(ii), len(cols)), np.float32)
    ncpd = len(CPD)
    with mp.get_context("fork").Pool(n_jobs, initializer=_init, initargs=(tables,)) as pp:
        for k in range(0, len(ii), chunk):
            i, j = ii[k:k + chunk], jj[k:k + chunk]
            A = {c: np.asarray(S1[c].take(i), dtype=object) for c in need}
            B = {c: np.asarray(PL[c].take(j), dtype=object) for c in need}
            f = cpdist_features(A, B)
            for q, (name, *_) in enumerate(CPD):
                out[k:k + len(i), q] = f[name]
            parts = [[A[c][q:q + sub].tolist() for c in PY_SRC] + [B[c][q:q + sub].tolist() for c in PY_SRC]
                     for q in range(0, len(i), sub)]
            out[k:k + len(i), ncpd:] = np.vstack(pp.map(_py_chunk, parts))
            del A, B, f, parts
    return pd.DataFrame(out, columns=cols, index=cands.index, copy=False)


# ------------------------------------------------------------------ competition context
def _top2(g, v):
    """Per-row: best and second-best value within the row's group (second=-1 if none)."""
    order = np.lexsort((-v, g))
    gs, vs = g[order], v[order]
    first = np.r_[True, gs[1:] != gs[:-1]]
    grp = np.cumsum(first) - 1
    starts = np.flatnonzero(first)
    sizes = np.diff(np.r_[starts, len(g)])
    b1 = vs[starts]
    b2 = np.where(sizes > 1, vs[np.minimum(starts + 1, len(g) - 1)], -1.0)
    t1, t2 = np.empty(len(g), vs.dtype), np.empty(len(g), vs.dtype)
    t1[order], t2[order] = b1[grp], b2[grp]
    return t1, t2


CTX = ["rank_s1", "gap_s1", "ratio_s1", "top2gap_s1", "rank_pool", "gap_pool", "margin_pool"]


def _context_into(out, col0, g1, g2, v, rows):
    """Write the 7 competition-context features of score v into out[:, col0:col0+7] (rows subset)."""
    from .blocking import _group_rank
    sel = (lambda x: x) if rows is None else (lambda x: x[rows])
    out[:, col0 + 0] = sel(_group_rank(g1, v))
    t1, t2 = _top2(g1, v)
    vv = sel(v)
    out[:, col0 + 1] = sel(t1) - vv
    out[:, col0 + 2] = vv / (sel(t1) + 1e-6)
    out[:, col0 + 3] = sel(t1) - sel(t2)
    del t1, t2
    out[:, col0 + 4] = sel(_group_rank(g2, v))
    p1, p2 = _top2(g2, v)
    p1s, p2s = sel(p1), sel(p2)
    del p1, p2
    out[:, col0 + 5] = p1s - vv
    out[:, col0 + 6] = vv - np.where(vv >= p1s, p2s, p1s)


def context_features(c, score_col, prefix):
    v = c[score_col].values.astype(np.float32)
    out = np.empty((len(c), len(CTX)), np.float32)
    _context_into(out, 0, c.s1_idx.values, c.pool_idx.values, v, None)
    return pd.DataFrame(out, columns=[f"{prefix}_{k}" for k in CTX], index=c.index)


BASE_CHEAP = ["bs_name", "bs_addr", "bs_cross", "bs_nkeys", "bs_nnk", "bs_nak", "bs_nxk", "pr_score", "pr_rank"]


def cheap_features(c, s1, pool, rows=None):
    """Features that need the FULL candidate set (competition context), materialized only for `rows`
    (bool mask or index array; None = all). One preallocated float32 block, no DataFrame copies:
    at 77M candidate pairs this is the difference between ~5 GB and an OOM kill."""
    if rows is not None and np.asarray(rows).dtype == bool:
        rows = np.flatnonzero(rows)
    n = len(c) if rows is None else len(rows)
    sel = (lambda x: x) if rows is None else (lambda x: x[rows])
    base = [b for b in BASE_CHEAP if b in c]
    has_pr = "pr_score" in c
    cols = (["src", "n_cands_s1", "n_cands_pool", "a1_empty", "a2_empty"] + base + ["bs_total"]
            + [f"bst_{k}" for k in CTX] + ([f"prs_{k}" for k in CTX] if has_pr else []))
    out = np.empty((n, len(cols)), np.float32)
    g1, g2 = c.s1_idx.values, c.pool_idx.values
    si, pj = sel(g1), sel(g2)
    out[:, 0] = pool.src.values[pj]
    out[:, 1] = np.bincount(g1, minlength=len(s1))[si]
    out[:, 2] = np.bincount(g2, minlength=len(pool))[pj]
    out[:, 3] = (np.asarray(s1.a_clean.str.len().fillna(0).values) == 0)[si]
    out[:, 4] = (np.asarray(pool.a_clean.str.len().fillna(0).values) == 0)[pj]
    k = 5
    for b in base:
        out[:, k] = sel(c[b].values)
        k += 1
    tot = (c.bs_name.values + c.bs_addr.values + c.bs_cross.values).astype(np.float32)
    out[:, k] = sel(tot)
    k += 1
    _context_into(out, k, g1, g2, tot, rows)
    del tot
    k += len(CTX)
    if has_pr:
        _context_into(out, k, g1, g2, c.pr_score.values.astype(np.float32), rows)
    return pd.DataFrame(out, columns=cols, copy=False)

In [ ]:
%%writefile src/matching.py
"""Pair classifier, set selection, and the competition metric."""
import numpy as np
import pandas as pd
import lightgbm as lgb

NON_FEATS = {"y", "s1_idx", "pool_idx", "fold"}

LGB_PARAMS = dict(objective="binary", learning_rate=0.08, num_leaves=127, min_data_in_leaf=100,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  max_bin=255, verbose=-1, num_threads=0)


def feat_cols(X):
    return [c for c in X.columns if c not in NON_FEATS]


def train_lgb(X, y, rounds=600, params=None, valid=None, feature_name="auto"):
    p = dict(LGB_PARAMS, **(params or {}))
    dtr = lgb.Dataset(X, y, free_raw_data=True, feature_name=feature_name)
    vs = [lgb.Dataset(valid[0], valid[1], reference=dtr)] if valid is not None else []
    cb = [lgb.log_evaluation(100)] + ([lgb.early_stopping(50, verbose=False)] if vs else [])
    return lgb.train(p, dtr, num_boost_round=rounds, valid_sets=vs, callbacks=cb)


def select(s1_idx, pool_idx, p, thr=0.5, one_to_one=True, min_top=None):
    """Pick matches. Each pool record goes to at most one S1 (its best-scoring one).
    `min_top`: if set, an S1 whose best candidate has p >= min_top keeps it even if < thr."""
    df = pd.DataFrame({"s1": s1_idx, "pool": pool_idx, "p": p})
    if one_to_one:
        best = df.groupby("pool").p.transform("max")
        df = df[df.p >= best]
        df = df.drop_duplicates("pool")
    keep = df.p >= thr
    if min_top is not None:
        top = df.groupby("s1").p.transform("max")
        keep |= (df.p >= top) & (df.p >= min_top)
    return df[keep][["s1", "pool", "p"]]


def f05_macro(pred_pairs, true_pairs, all_s1):
    """pred_pairs/true_pairs: DataFrames with columns s1, pool (ids or idx). all_s1: iterable of s1 keys."""
    all_s1 = pd.Index(pd.unique(np.asarray(all_s1)))
    tp = pred_pairs.merge(true_pairs, on=["s1", "pool"]).groupby("s1").size()
    npred = pred_pairs.groupby("s1").size()
    ntrue = true_pairs.groupby("s1").size()
    tp, npred, ntrue = (s.reindex(all_s1, fill_value=0).values.astype(float) for s in (tp, npred, ntrue))
    with np.errstate(divide="ignore", invalid="ignore"):
        prec = np.where(npred > 0, tp / npred, 0.0)
        rec = np.where(ntrue > 0, tp / ntrue, 0.0)
        f = np.where(tp > 0, 1.25 * prec * rec / (0.25 * prec + rec), 0.0)
    f = np.where((ntrue == 0) & (npred == 0), 1.0, f)
    return float(f.mean()), dict(precision=float(prec[npred > 0].mean()) if (npred > 0).any() else 0.0,
                                 recall=float(rec[ntrue > 0].mean()),
                                 singleton_acc=float(((npred == 0)[ntrue == 0]).mean()) if (ntrue == 0).any() else 1.0)

In [ ]:
%%writefile src/noise.py
"""Learned noise-model features.

The data generator perturbs true copies and distractors differently:
  * distractors ADD tokens from a specific vocabulary (public, private, group, holdings, india, ...)
    and SUBSTITUTE house-number digits (111 -> 115);
  * true copies add alias markers (dba, formerly, aka, nee), drop/mask/zero-pad digits (16282 -> 6282, ##1).
We learn per-token log-odds of being ADDED / DROPPED in true pairs vs. negative candidates (from
training labels, out-of-fold) and classify the edit relation between house numbers.
"""
import re
from collections import Counter

import numpy as np
from unidecode import unidecode

from .normalize import skeleton, collapse, LEGAL

SK = "\u00a7"   # prefix for phonetic-stem keys inside the log-odds dicts


from functools import lru_cache


@lru_cache(maxsize=2_000_000)
def stem_key(t: str):
    """Language-agnostic stem: singularize + consonant skeleton ('groupe'/'group' -> 'jrp',
    'developpement'/'development' -> 'dblpmnt'). None when too short to be distinctive."""
    if len(t) > 4 and t.endswith("s"):
        t = t[:-1]
    k = skeleton(collapse(t))
    return k if len(k) >= 3 else None


LEGAL_KEY, COUNTRY_KEY = SK + "LEGAL", SK + "COUNTRY"


def _special_key(t, country):
    if country and t == country:
        return COUNTRY_KEY
    if t in LEGAL and LEGAL[t]:
        return LEGAL_KEY
    return None


def lookup(lo, t, country="", freq=None):
    """exact token -> [only for corpus-frequent tokens] special class (own country name / legal form)
    -> phonetic stem -> 0. Frequency gate: distractor markers come from a vocabulary (frequent),
    typos are random (rare) and must not inherit a stem's statistics."""
    v = lo.get(t)
    if v is not None:
        return v
    if freq is None or t not in freq:
        return 0.0
    sp = _special_key(t, country)
    if sp:
        return lo.get(sp, 0.0)
    k = stem_key(t)
    return lo.get(SK + k, 0.0) if k else 0.0


_TOK = re.compile(r"[a-z0-9]+")
_NUM = re.compile(r"[0-9#]+")


def raw_tokens(s: str):
    if not s:
        return ()
    if not s.isascii():
        s = unidecode(s)
    return tuple(sorted(set(_TOK.findall(s.lower()))))


def raw_numbers(s: str):
    if not s:
        return ()
    return tuple(t for t in _NUM.findall(s) if any(c.isdigit() for c in t))


# ------------------------------------------------------------------ token log-odds
class LogOddsCounter:
    """Incremental version of learn_logodds (feed chunks, then .result())."""

    def __init__(self):
        self.add_p, self.add_n, self.drop_p, self.drop_n = Counter(), Counter(), Counter(), Counter()
        self.npos = self.nneg = 0

    def update(self, pairs_a, pairs_b, y, countries=None):
        countries = countries if countries is not None else [""] * len(y)
        for a, b, t, ct in zip(pairs_a, pairs_b, y, countries):
            sa, sb = set(a.split()), set(b.split())
            add, drop = sb - sa, sa - sb
            ct = (ct or "").lower()
            add = list(add) + [k for k in (_special_key(x, ct) for x in add) if k]
            drop = list(drop) + [k for k in (_special_key(x, ct) for x in drop) if k]
            if t:
                self.npos += 1
                self.add_p.update(add)
                self.drop_p.update(drop)
            else:
                self.nneg += 1
                self.add_n.update(add)
                self.drop_n.update(drop)

    def result(self, min_count=25, prior=1.0):
        npos, nneg = self.npos, self.nneg

        def lo(cp, cn):
            out = {k: float(np.log((cp[k] + prior) / (npos + prior)) - np.log((cn[k] + prior) / (nneg + prior)))
                   for k in set(cp) | set(cn) if cp[k] + cn[k] >= min_count}
            # phonetic-stem aggregate: lets unseen spellings in another language ('groupe') inherit
            # the statistics of their cognate ('group'); used only when the exact token is unknown
            sp, sn = Counter(), Counter()
            for c_, d_ in ((cp, sp), (cn, sn)):
                for k, v in c_.items():
                    if k.startswith(SK):
                        continue
                    sk = stem_key(k)
                    if sk:
                        d_[sk] += v
            for k in set(sp) | set(sn):
                if sp[k] + sn[k] >= min_count:
                    out[SK + k] = float(np.log((sp[k] + prior) / (npos + prior)) - np.log((sn[k] + prior) / (nneg + prior)))
            return out

        return lo(self.add_p, self.add_n), lo(self.drop_p, self.drop_n)


def learn_logodds(pairs_a, pairs_b, y, min_count=25, prior=1.0):
    """pairs_a/pairs_b: sequences of token tuples (S1 side, pool side). y: labels.
    Returns (add_lo, drop_lo) dicts: log P(token added | match) / P(token added | non-match)."""
    add_p, add_n, drop_p, drop_n = Counter(), Counter(), Counter(), Counter()
    npos = nneg = 0
    for a, b, t in zip(pairs_a, pairs_b, y):
        sa, sb = set(a), set(b)
        if t:
            npos += 1
            add_p.update(sb - sa)
            drop_p.update(sa - sb)
        else:
            nneg += 1
            add_n.update(sb - sa)
            drop_n.update(sa - sb)

    def lo(cp, cn):
        out = {}
        for k in set(cp) | set(cn):
            if cp[k] + cn[k] >= min_count:
                out[k] = float(np.log((cp[k] + prior) / (npos + prior)) - np.log((cn[k] + prior) / (nneg + prior)))
        return out

    return lo(add_p, add_n), lo(drop_p, drop_n)


def token_noise_feats(a, b, add_lo, drop_lo, country="", freq=None):
    sa, sb = set(a), set(b)
    add_t, drop_t = sb - sa, sa - sb
    add = [lookup(add_lo, t, country, freq) for t in add_t]
    drop = [lookup(drop_lo, t, country, freq) for t in drop_t]
    return (min(add, default=0.0), sum(add), max(add, default=0.0), float(sum(v < -2 for v in add)),
            min(drop, default=0.0), sum(drop), float(len(add_t)), float(len(drop_t)))


def frequent_tokens(series_list, min_df=50):
    c = Counter()
    for s_ in series_list:
        for v in s_:
            if v:
                c.update(v.split())
    return frozenset(k for k, n in c.items() if n >= min_df)


TOKEN_COLS = ["add_min", "add_sum", "add_max", "add_nbad", "drop_min", "drop_sum", "n_add", "n_drop"]

# ------------------------------------------------------------------ number edit relation
REL = ["equal", "zero_pad", "masked", "deletion", "insertion", "transpose", "substitution", "unrelated", "missing"]
_RANK = {r: i for i, r in enumerate(REL)}


def _rel(x, y):
    if x == y:
        return 0
    if x.lstrip("0") == y.lstrip("0"):
        return 1
    if "#" in y:
        core = y.replace("#", "")
        if (len(y) == len(x) and all(c == d or c == "#" for d, c in zip(x, y))) or (core and core.lstrip("0") in x):
            return 2
    lx, ly = len(x), len(y)
    if ly == lx - 1 and any(x[:i] + x[i + 1:] == y for i in range(lx)):
        return 3
    if ly == lx + 1 and any(y[:i] + y[i + 1:] == x for i in range(ly)):
        return 4
    if lx == ly:
        d = sum(c != e for c, e in zip(x, y))
        if d == 2 and sorted(x) == sorted(y):
            return 5
        if d == 1:
            return 6
    return 7


def number_feats(a, b):
    """a: S1 numbers, b: pool numbers. Returns best relation, counts of equal/substitution, coverage."""
    if not a or not b:
        return (8.0, 0.0, 0.0, 0.0, 0.0, float(len(a)), float(len(b)))
    best_for_a = [min(_rel(x, y) for y in b) for x in a]
    best = min(best_for_a)
    n_eq = sum(r <= 1 for r in best_for_a)
    n_soft = sum(2 <= r <= 5 for r in best_for_a)
    n_sub = sum(r == 6 for r in best_for_a)
    first = min(_rel(a[0], y) for y in b)
    return (float(best), float(first), n_eq / len(a), n_soft / len(a), n_sub / len(a), float(len(a)), float(len(b)))


NUM_COLS = ["num_rel_best", "num_rel_first", "num_eq_frac", "num_soft_frac", "num_sub_frac", "num_n1", "num_n2"]

In [ ]:
%%writefile src/normalize.py
"""Text normalization for business names and addresses.

Country-agnostic: every rule is language-level (Latin transliteration, punctuation,
abbreviation folding), never keyed on a fixed country list, so unseen countries
(France in test) go through the same path.
"""
import re
from unidecode import unidecode

# ----------------------------------------------------------------------------- tokens
LEGAL = {
    # EN / IN
    "pvt": "pvt", "private": "pvt", "pte": "pvt", "prvt": "pvt",
    "ltd": "ltd", "limited": "ltd", "ltda": "ltd", "lt": "ltd",
    "llc": "llc", "llp": "llp", "lp": "lp", "plc": "plc", "pllc": "pllc", "pc": "pc",
    "inc": "inc", "incorporated": "inc", "corp": "corp", "corporation": "corp",
    "co": "co", "company": "co", "cos": "co", "tbk": "tbk", "gmbh": "gmbh", "ag": "ag",
    "public": "public", "opc": "opc", "holdings": None, "group": None,
    # FR (and generic civil-law)
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "eurl": "eurl", "sa": "sa", "sci": "sci",
    "snc": "snc", "sca": "sca", "scop": "scop", "selarl": "selarl", "scp": "scp", "gie": "gie",
}
LEGAL_FORMS = {v for v in LEGAL.values() if v}
NAME_STOP = {"the", "and", "of", "de", "des", "du", "la", "le", "les", "d", "l", "et", "a",
             "an", "www", "com", "net", "org", "in", "co.in", "biz", "info", "fr", "us", "dba",
             "nee", "id", "m/s", "ms", "mr", "mrs", "sri", "shri", "shree"}

ADDR_MAP = {
    "street": "st", "str": "st", "st": "st", "avenue": "ave", "av": "ave", "ave": "ave", "avn": "ave",
    "road": "rd", "rd": "rd", "drive": "dr", "dr": "dr", "lane": "ln", "ln": "ln",
    "circle": "cir", "cir": "cir", "court": "ct", "ct": "ct", "boulevard": "blvd", "blvd": "blvd",
    "bd": "blvd", "bld": "blvd", "place": "pl", "pl": "pl", "parkway": "pkwy", "pkwy": "pkwy",
    "highway": "hwy", "hwy": "hwy", "terrace": "ter", "ter": "ter", "trail": "trl", "trl": "trl",
    "square": "sq", "sq": "sq", "cove": "cv", "cv": "cv", "way": "way", "loop": "loop",
    "north": "n", "south": "s", "east": "e", "west": "w", "northeast": "ne", "northwest": "nw",
    "southeast": "se", "southwest": "sw", "suite": "ste", "ste": "ste", "apartment": "apt", "apt": "apt",
    "floor": "flr", "flr": "flr", "fl": "flr", "etage": "flr", "building": "bldg", "bldg": "bldg",
    "sector": "sec", "sec": "sec", "nagar": "ngr", "ngr": "ngr", "colony": "col", "marg": "marg",
    # FR
    "rue": "rue", "r": "rue", "allee": "all", "all": "all", "chemin": "chem", "chem": "chem",
    "impasse": "imp", "imp": "imp", "route": "rte", "rte": "rte", "faubourg": "fbg", "fbg": "fbg",
    "quai": "quai", "cours": "crs", "crs": "crs", "residence": "res", "res": "res",
}
ADDR_STOP = {"no", "num", "number", "nr", "of", "the", "de", "des", "du", "la", "le", "les", "d", "l",
             "city", "cdp", "borough", "village", "town", "township", "bis", "ter", "unit", "near",
             "opp", "opposite", "behind", "landmark", "house", "door", "flat", "plot", "shop", "h",
             "hno", "dno", "p", "s/o", "so", "at", "po", "post", "dist", "district", "tehsil", "taluka",
             "eme", "er", "nd", "rd_", "th"}

# state / region names -> canonical code, matched by phonetic skeleton so that
# "Maharashtra", "महाराष्ट्र" (-> "mhaaraassttr") and typos land on the same code.
_STATES = {
    # US
    "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california",
    "co": "colorado", "ct": "connecticut", "de": "delaware", "dc": "district of columbia",
    "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho", "il": "illinois",
    "in": "indiana", "ia": "iowa", "ks": "kansas", "ky": "kentucky", "la": "louisiana",
    "me": "maine", "md": "maryland", "ma": "massachusetts", "mi": "michigan", "mn": "minnesota",
    "ms": "mississippi", "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york",
    "nc": "north carolina", "nd": "north dakota", "oh": "ohio", "ok": "oklahoma", "or": "oregon",
    "pa": "pennsylvania", "ri": "rhode island", "sc": "south carolina", "sd": "south dakota",
    "tn": "tennessee", "tx": "texas", "ut": "utah", "vt": "vermont", "va": "virginia",
    "wa": "washington", "wv": "west virginia", "wi": "wisconsin", "wy": "wyoming",
}
_IN_STATES = {
    "mh": "maharashtra", "dl": "delhi", "up": "uttar pradesh", "ka": "karnataka", "tn": "tamil nadu",
    "gj": "gujarat", "wb": "west bengal", "tg": "telangana", "ts": "telangana", "hr": "haryana",
    "kl": "kerala", "rj": "rajasthan", "br": "bihar", "mp": "madhya pradesh", "ap": "andhra pradesh",
    "od": "orissa", "or": "odisha", "pb": "punjab", "jh": "jharkhand", "ch": "chandigarh",
    "ga": "goa", "as": "assam", "uk": "uttarakhand", "hp": "himachal pradesh", "jk": "jammu and kashmir",
    "ct": "chhattisgarh", "cg": "chhattisgarh", "py": "puducherry",
}

# ----------------------------------------------------------------------------- helpers
_RE_ID = re.compile(r"\(\s*id\s*:?\s*\d+\s*\)", re.I)
_RE_DOT_SPLIT = re.compile(r"\.(?=[a-z0-9]{2,})")
_RE_NONALNUM = re.compile(r"[^a-z0-9/ ]+")
_RE_SPACES = re.compile(r"\s+")
_RE_ORD = re.compile(r"\b(\d+)(st|nd|rd|th|eme|er|e)\b")
_RE_NUM = re.compile(r"\d+")
_OCR = str.maketrans({"0": "o", "1": "l", "5": "s", "3": "e", "4": "a", "8": "b", "7": "t"})


def to_ascii(s: str) -> str:
    if not s:
        return ""
    if not s.isascii():
        s = unidecode(s.replace("\u00b0", " "))
    return s.lower()


def _basic(s: str) -> str:
    s = to_ascii(s)
    s = _RE_ID.sub(" ", s)
    s = s.replace("&", " and ").replace("+", " plus ").replace("@", " at ")
    s = _RE_DOT_SPLIT.sub(" ", s)      # "pvt.ltd" -> "pvt ltd", "ironl.com" -> "ironl com"
    s = s.replace(".", "")             # "s.a.r.l." -> "sarl", "inc." -> "inc"
    s = s.replace("'", "")
    s = _RE_NONALNUM.sub(" ", s)
    return _RE_SPACES.sub(" ", s).strip()


def collapse(t: str) -> str:
    out = []
    for c in t:
        if not out or out[-1] != c:
            out.append(c)
    return "".join(out)


_DIGRAPHS = [("ph", "f"), ("bh", "b"), ("kh", "k"), ("gh", "g"), ("th", "t"), ("dh", "d"),
             ("sh", "s"), ("ch", "c"), ("ck", "k"), ("nb", "mb"), ("np", "mp"), ("x", "ks"),
             ("q", "k"), ("c", "k"), ("v", "b"), ("w", ""), ("z", "j"), ("g", "j"), ("h", "")]
_VOWELS = set("aeiouy")


def skeleton(t: str) -> str:
    """Phonetic consonant skeleton; makes transliterations / vowel typos collide.
    'private'/'praaivett'/'praibhet' -> 'prbt';  'management'/'myaanejmentt' -> 'mnjmnt'."""
    t = collapse(t)
    for a, b in _DIGRAPHS:
        if a in t:
            t = t.replace(a, b)
    return collapse("".join(c for c in t if c not in _VOWELS))


def _fix_ocr(tok: str) -> str:
    # tokens mixing letters and digits with a majority of letters: '0ne' -> 'one'
    if tok.isalnum() and not tok.isalpha() and not tok.isdigit():
        n_alpha = sum(c.isalpha() for c in tok)
        if n_alpha >= len(tok) / 2:
            return tok.translate(_OCR)
    return tok


_STATE_SKEL = {}
for _code, _name in list(_STATES.items()) + list(_IN_STATES.items()):
    _STATE_SKEL.setdefault(skeleton(_name.replace(" ", "")), _code)


def state_of(part: str):
    """Map one comma-separated address component to a state code if it is one."""
    p = part.replace(" ", "")
    if len(p) == 2 and p.isalpha():
        return p if (p in _STATES or p in _IN_STATES) else None
    return _STATE_SKEL.get(skeleton(p)) if len(p) >= 4 else None


# ----------------------------------------------------------------------------- public API
_LEGAL_SKEL = {skeleton(k): v for k, v in LEGAL.items() if v and len(k) >= 5}


def _legal(t: str):
    """Legal-form lookup that survives transliteration ('praivet', 'limittedd')."""
    if t in LEGAL:
        return LEGAL[t]
    if len(t) >= 5:
        c = collapse(t)
        if c in LEGAL:
            return LEGAL[c]
        return _LEGAL_SKEL.get(skeleton(t))
    return None


def norm_name(s: str):
    """Returns (clean_string, core_tokens, legal_forms)."""
    b = _basic(s)
    toks = [_fix_ocr(t) for t in b.replace("/", " ").split()]
    core, legal = [], []
    for t in toks:
        v = _legal(t)
        if v:
            legal.append(v)
            continue
        if t in NAME_STOP:
            continue
        core.append(collapse(t) if len(t) > 3 else t)
    return " ".join(core), core, sorted(set(legal))


def norm_addr(s: str):
    """Returns (clean_string, alpha_tokens, numbers, state_code)."""
    if not s:
        return "", [], [], ""
    a = to_ascii(s).replace("#", " ")
    state, keep = "", []
    for part in a.split(","):
        p = _RE_NONALNUM.sub(" ", part.replace(".", " ")).strip()
        st = state_of(p) if p else None
        if st:
            state = st
        else:
            keep.append(part)
    b = _basic(",".join(keep))
    b = _RE_ORD.sub(r"\1", b)
    b = b.replace("/", " ")
    alpha, nums = [], []
    for t in b.split():
        if t.isdigit():
            t = t.lstrip("0") or "0"
            nums.append(t)
            continue
        t = _fix_ocr(t)
        if any(c.isdigit() for c in t):          # 1715b, 8c5 -> keep leading number too
            m = _RE_NUM.match(t)
            if m:
                nums.append(m.group(0).lstrip("0") or "0")
            alpha.append(t)
            continue
        t = ADDR_MAP.get(t, t)
        if t in ADDR_STOP or len(t) < 2:
            continue
        alpha.append(collapse(t))
    return " ".join(alpha), alpha, nums, state

In [ ]:
%%writefile src/prep.py
"""Load the three sources, normalize in parallel, cache as parquet."""
import os
import multiprocessing as mp
import numpy as np
import pandas as pd

from .normalize import norm_name, norm_addr, skeleton
from .noise import raw_tokens, raw_numbers

NORM_COLS = ["n_clean", "legal", "a_clean", "nums", "state", "nsk", "ask", "rn", "ra", "ru"]
READ_KW = dict(sep="\t", dtype=str, keep_default_na=False, quoting=3)


def read_tsv(path):
    df = pd.read_csv(path, **READ_KW)
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        df[c] = df[c].str.replace("\r", "", regex=False)
    return df


def _norm_chunk(args):
    names, addrs = args
    out = []
    for n, a in zip(names, addrs):
        nc, _, legal = norm_name(n)
        ac, _, nums, st = norm_addr(a)
        out.append((nc, " ".join(legal), ac, " ".join(nums), st,
                    " ".join(skeleton(t) for t in nc.split()),          # name skeleton tokens
                    " ".join(skeleton(t) if t.isalpha() else t for t in ac.split()),  # addr skeleton tokens
                    " ".join(raw_tokens(n)), " ".join(raw_tokens(a)),   # raw tokens (noise model)
                    " ".join(raw_numbers(a))))                          # raw numbers incl. '#' masks
    return out


def normalize_df(df, n_jobs=None, chunk=50_000):
    n_jobs = n_jobs or os.cpu_count()
    names, addrs = df.business_name.tolist(), df.business_address.tolist()
    parts = [(names[i:i + chunk], addrs[i:i + chunk]) for i in range(0, len(df), chunk)]
    if n_jobs > 1 and len(parts) > 1:
        with mp.get_context("fork").Pool(n_jobs) as pool:
            res = pool.map(_norm_chunk, parts)
    else:
        res = [_norm_chunk(p) for p in parts]
    flat = [r for part in res for r in part]
    out = pd.DataFrame(flat, columns=NORM_COLS, index=df.index)
    # raw text is no longer needed downstream; dropping it roughly halves RAM at full scale
    return pd.concat([df.drop(columns=["business_name", "business_address"]), out], axis=1)


def _compact(df):
    """Arrow-backed strings: ~3x less RAM than Python str objects at 10M rows."""
    for c in NORM_COLS:
        if c in df:
            df[c] = df[c].astype("string[pyarrow]")
    return df


def _read_arrow(path):
    """Read parquet straight into Arrow-backed string columns (never materializes Python str objects)."""
    import pyarrow.parquet as pq
    t = pq.read_table(path)
    df = t.to_pandas(types_mapper=lambda typ: pd.StringDtype("pyarrow") if (str(typ) in ("string", "large_string")) else None,
                     self_destruct=True)
    del t
    return df


def load_split(data_dir, split, cache_dir=None, n_jobs=None):
    """Returns (s1, pool). pool = S2 ++ S3 with a `src` column (2/3)."""
    cache = os.path.join(cache_dir, f"{split}_norm") if cache_dir else None
    if cache and os.path.exists(cache + "_s1.parquet"):
        return _read_arrow(cache + "_s1.parquet"), _read_arrow(cache + "_pool.parquet")
    d = os.path.join(data_dir, split)
    s1 = read_tsv(os.path.join(d, f"{split}_source1.tsv"))
    s2 = read_tsv(os.path.join(d, f"{split}_source2.tsv"))
    s3 = read_tsv(os.path.join(d, f"{split}_source3.tsv"))
    s2["src"], s3["src"] = np.int8(2), np.int8(3)
    pool = pd.concat([s2, s3], ignore_index=True)
    s1["src"] = np.int8(1)
    s1 = normalize_df(s1, n_jobs)
    pool = normalize_df(pool, n_jobs)
    if cache:
        os.makedirs(cache_dir, exist_ok=True)
        s1.to_parquet(cache + "_s1.parquet")
        pool.to_parquet(cache + "_pool.parquet")
    return _compact(s1), _compact(pool)


def load_gt(path):
    gt = read_tsv(path)
    rows = [(s, m) for s, ms in zip(gt.source1_entity_id, gt.matched_entity_ids) for m in ms.split(",") if m]
    return gt, pd.DataFrame(rows, columns=["s1_id", "pool_id"])

In [ ]:
%%writefile src/prerank.py
"""Learned candidate pruning (stage 0): cheap blocking-level features only (S1-side context),
so it can run inside every blocking chunk."""
import numpy as np
import pandas as pd

from .blocking import _group_rank

PR_COLS = ["bs_name", "bs_addr", "bs_cross", "bs_nkeys", "bs_nnk", "bs_nak", "bs_nxk", "bs_total",
           "r_total", "r_name", "r_addr", "r_cross", "q_total", "q_name", "q_addr", "q_cross", "n_s1"]


def prerank_features(b):
    g = b.s1_idx.values
    X = pd.DataFrame({c: b[c].values.astype(np.float32) for c in
                      ["bs_name", "bs_addr", "bs_cross", "bs_nkeys", "bs_nnk", "bs_nak", "bs_nxk"]})
    X["bs_total"] = X.bs_name + X.bs_addr + X.bs_cross
    for k, c in [("total", "bs_total"), ("name", "bs_name"), ("addr", "bs_addr"), ("cross", "bs_cross")]:
        v = X[c].values
        X["r_" + k] = _group_rank(g, v).astype(np.float32)
        mx = pd.Series(v).groupby(g).transform("max").values
        X["q_" + k] = (v / (mx + 1e-6)).astype(np.float32)
    X["n_s1"] = pd.Series(g).groupby(g).transform("size").values.astype(np.float32)
    return X[PR_COLS]


def make_pruner(model, k_keep, p_keep=None):
    """Keep top-k_keep per S1 by pre-rank score (and anything with score >= p_keep)."""
    def prune(b):
        if len(b) == 0:
            return b
        s = model.predict(prerank_features(b), num_threads=0).astype(np.float32)
        r = _group_rank(b.s1_idx.values, s)
        keep = r < k_keep
        if p_keep is not None:
            keep |= s >= p_keep
        out = b[keep].copy()
        out["pr_score"] = s[keep]
        out["pr_rank"] = r[keep].astype(np.float32)
        return out
    return prune

In [ ]:
%%writefile src/run.py
"""End-to-end pipeline v2.

data -> normalize -> key index (cap 1200 single / 600 composite)
     -> blocking pass on a small S1 slice (wide K) -> train pre-ranker
     -> blocking over ALL S1 with learned pruning (top-K per S1)          [train and test]
     -> noise-model dictionaries learned on S1 group D (labels, out-of-model-sample)
     -> features on S1 group M -> LightGBM (holdout = part of M) -> threshold on holdout macro F0.5
     -> test features in chunks -> predict -> one-to-one selection -> TSVs

Train S1 roles by hash h in [0, 10000):
  h <  PR_END            : pre-ranker training slice (blocking-level labels only)
  h in [M_START, M_END)  : model training (last `holdout` share = holdout)
  h >= D_START           : noise-dictionary learning
Every stage caches to --work; delete a file there to redo it.
"""
import argparse
import gc
import json
import os
import pickle
import time

import numpy as np
import pandas as pd
import lightgbm as lgb

from .prep import load_split, load_gt
from .blocking import KeyIndex, generate_candidates
from .prerank import prerank_features, make_pruner, PR_COLS
from .features import build_idf, cheap_features, string_features
from .noise import LogOddsCounter, frequent_tokens
from .matching import train_lgb, feat_cols, select, f05_macro

T0 = time.time()


def rss_gb():
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS"):
                    return int(line.split()[1]) / 1e6
    except OSError:
        pass
    return -1.0


def log(*a):
    print(f"[{time.time() - T0:7.0f}s | {rss_gb():5.1f}GB]", *a, flush=True)


def s1_hash(ids):
    return (pd.util.hash_array(np.asarray(ids, dtype=object)) % 10_000).astype(np.int32)


def labels(c, s1, pool, gtp):
    """Integer-key join (no 77M-row string MultiIndex): pair key = s1_idx * 2^24 + pool_idx."""
    si = pd.Index(s1.entity_id.values).get_indexer(gtp.s1_id.values)
    pj = pd.Index(pool.entity_id.values).get_indexer(gtp.pool_id.values)
    ok = (si >= 0) & (pj >= 0)
    gk = np.unique((si[ok].astype(np.int64) << 24) | pj[ok].astype(np.int64))
    ck = (c.s1_idx.values.astype(np.int64) << 24) | c.pool_idx.values.astype(np.int64)
    return np.isin(ck, gk, assume_unique=False).astype(np.int8)


def write_lists(path, s1_ids, s1_idx, pool_ids_col, pool_idx, col, chunk=200_000):
    """Stream one row per S1 (empty list allowed). Works on integer indices; strings are only
    materialized per chunk of S1 rows. Returns a bool array: S1 row has >= 1 id."""
    s1_ids = np.asarray(s1_ids, dtype=object)
    key = (np.asarray(s1_idx, np.int64) << 24) | np.asarray(pool_idx, np.int64)
    key = np.unique(key)                       # sorted by s1, dedup
    ks, kp = (key >> 24).astype(np.int64), (key & ((1 << 24) - 1)).astype(np.int64)
    bounds = np.searchsorted(ks, np.arange(len(s1_ids) + 1))
    has = np.diff(bounds) > 0
    with open(path, "w") as f:
        f.write(f"source1_entity_id\t{col}\n")
        for r0 in range(0, len(s1_ids), chunk):
            r1 = min(r0 + chunk, len(s1_ids))
            b0, b1 = bounds[r0], bounds[r1]
            pid = np.asarray(pool_ids_col.take(kp[b0:b1]), dtype=object)
            lines = []
            for r in range(r0, r1):
                lo, hi = bounds[r] - b0, bounds[r + 1] - b0
                lines.append(f"{s1_ids[r]}\t{','.join(pid[lo:hi])}\n")
            f.write("".join(lines))
    return has


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--work", default="work")
    ap.add_argument("--out", default="output")
    ap.add_argument("--cap", type=int, default=1200, help="pool df cap for single-token keys")
    ap.add_argument("--cap-composite", type=int, default=600, help="pool df cap for composite keys")
    ap.add_argument("--df-scale", type=float, default=1.0)
    ap.add_argument("--wide", default="150,40,40,40", help="k_total,k_name,k_addr,k_cross before pruning")
    ap.add_argument("--k-keep", type=int, default=35, help="candidates kept per S1 after pre-ranking")
    ap.add_argument("--pr-frac", type=float, default=0.05)
    ap.add_argument("--m-frac", type=float, default=0.15, help="share of train S1 used for the model")
    ap.add_argument("--holdout", type=float, default=0.2)
    ap.add_argument("--d-frac", type=float, default=0.55, help="share of train S1 used for noise dictionaries")
    ap.add_argument("--rounds", type=int, default=2000)
    ap.add_argument("--lr", type=float, default=0.1)
    ap.add_argument("--threshold", type=float, default=None)
    ap.add_argument("--predict-chunk", type=int, default=3_000_000)
    ap.add_argument("--jobs", type=int, default=None)
    ap.add_argument("--d-max-pairs", type=int, default=15_000_000, help="max candidate pairs for noise dictionaries")
    ap.add_argument("--stage", default="all", choices=["all", "train", "test-cands", "predict"],
                    help="train: stop after model.pkl | test-cands: pre-ranker + test blocking only | "
                         "predict: needs model.pkl, prerank.txt, test_cands.parquet in --work")
    ap.add_argument("--extra-work", nargs="*", default=[],
                    help="dirs whose files are copied into --work first (outputs of other jobs)")
    a = ap.parse_args()
    os.makedirs(a.work, exist_ok=True)
    os.makedirs(a.out, exist_ok=True)
    import shutil
    for d in a.extra_work:
        for f in os.listdir(d):
            src, dst = os.path.join(d, f), os.path.join(a.work, f)
            if os.path.isfile(src) and not os.path.exists(dst):
                os.symlink(src, dst) if f.endswith(".parquet") else shutil.copy(src, dst)
                log("imported", f, "from", d)
    W = lambda f: os.path.join(a.work, f)
    kt, kn, ka, kx = map(int, a.wide.split(","))
    wide = dict(k_total=kt, k_name=kn, k_addr=ka, k_cross=kx)
    PR_END = int(a.pr_frac * 10_000)
    M_START, M_END = PR_END, PR_END + int(a.m_frac * 10_000)
    D_START = 10_000 - int(a.d_frac * 10_000)
    assert D_START >= M_END, "dictionary S1 group must not overlap the model group"

    def ensure_prerank(s1, pool, gtp, h, ix):
        """Deterministic pre-ranker (same data + params -> same model in every job)."""
        if os.path.exists(W("prerank.txt")):
            return lgb.Booster(model_file=W("prerank.txt"))
        cw = generate_candidates(s1, pool, index=ix, s1_subset=h < PR_END, verbose=False, **wide)
        yw = labels(cw, s1, pool, gtp)
        ntrue = gtp.s1_id.isin(set(s1.entity_id.values[h < PR_END])).sum()
        log(f"pre-ranker slice: {len(cw):,} wide pairs, wide recall {yw.sum() / ntrue:.4f}")
        Xw = prerank_features(cw)
        prm = dict(objective="binary", learning_rate=0.1, num_leaves=63, min_data_in_leaf=200, verbose=-1,
                   num_threads=4, deterministic=True, force_row_wise=True, seed=7)
        from .blocking import _group_rank
        half = (cw.s1_idx.values % 2) == 0
        m0 = lgb.train(prm, lgb.Dataset(Xw[half], yw[half]), 200)
        r = _group_rank(cw.s1_idx.values[~half], m0.predict(Xw[~half]))
        nt_half = gtp.s1_id.isin(set(s1.entity_id.values[np.unique(cw.s1_idx.values[~half])])).sum()
        for K in (20, 30, 35, 40, 60):
            log(f"  pre-ranker recall@{K} {yw[~half][r < K].sum() / nt_half:.4f} (held-out half of slice)")
        pr = lgb.train(prm, lgb.Dataset(Xw, yw), 200)
        pr.save_model(W("prerank.txt"))
        return pr

    if a.stage == "test-cands" and not os.path.exists(W("prerank.txt")):
        s1, pool = load_split(a.data, "train", a.work, a.jobs)
        _, gtp = load_gt(os.path.join(a.data, "train", "train_ground_truth.tsv"))
        ix = KeyIndex(s1, pool, a.cap, a.df_scale, a.jobs, cap_composite=a.cap_composite)
        ensure_prerank(s1, pool, gtp, s1_hash(s1.entity_id.values), ix)
        del s1, pool, ix
        gc.collect()

    # ================================================================ TRAIN
    if a.stage in ("all", "train") and not os.path.exists(W("model.pkl")):
        s1, pool = load_split(a.data, "train", a.work, a.jobs)
        log("train loaded", len(s1), len(pool))
        _, gtp = load_gt(os.path.join(a.data, "train", "train_ground_truth.tsv"))
        h = s1_hash(s1.entity_id.values)

        if not os.path.exists(W("train_cands.parquet")):
            ix = KeyIndex(s1, pool, a.cap, a.df_scale, a.jobs, cap_composite=a.cap_composite)
            pr = ensure_prerank(s1, pool, gtp, h, ix)
            c = generate_candidates(s1, pool, index=ix, prerank=make_pruner(pr, a.k_keep), **wide)
            del ix
            gc.collect()
            c.to_parquet(W("train_cands.parquet"))
        c = pd.read_parquet(W("train_cands.parquet"))
        y_all = labels(c, s1, pool, gtp)
        log(f"train cands {len(c):,}  per S1 {len(c) / len(s1):.1f}  blocking recall {y_all.sum() / len(gtp):.4f}")

        # ---- noise-model dictionaries from group D (not used for model fitting)
        hc = h[c.s1_idx.values]
        d = hc >= D_START
        dsel = np.flatnonzero(d)
        if len(dsel) > a.d_max_pairs:   # plenty for token statistics; bounds time and memory
            dsel = np.sort(np.random.default_rng(0).choice(dsel, a.d_max_pairs, replace=False))
        cnt_n, cnt_a = LogOddsCounter(), LogOddsCounter()
        di_all, dj_all, dy_all = c.s1_idx.values[dsel], c.pool_idx.values[dsel], y_all[dsel]
        for k in range(0, len(dsel), 2_000_000):     # stream: never more than 2M pairs of strings alive
            di, dj, dy = di_all[k:k + 2_000_000], dj_all[k:k + 2_000_000], dy_all[k:k + 2_000_000]
            ct = s1.country.array.take(di)
            cnt_n.update(s1.rn.array.take(di), pool.rn.array.take(dj), dy, ct)
            cnt_a.update(s1.ra.array.take(di), pool.ra.array.take(dj), dy, ct)
        nadd, ndrop = cnt_n.result()
        aadd, adrop = cnt_a.result()
        del cnt_n, cnt_a, di_all, dj_all, dy_all, dsel, d
        gc.collect()
        log(f"noise dicts from {min(int((hc >= D_START).sum()), a.d_max_pairs):,} D pairs: name add {len(nadd)} drop {len(ndrop)} | addr add {len(aadd)} drop {len(adrop)}")
        idf_n = build_idf([s1.nsk, pool.nsk])
        idf_a = build_idf([s1.ask, pool.ask])
        fq = (frequent_tokens([s1.rn, pool.rn]), frequent_tokens([s1.ra, pool.ra]))
        tables = (*idf_n, *idf_a, nadd, ndrop, aadd, adrop, *fq)

        # ---- features for group M (context computed over ALL candidates, materialized for M only)
        m = (hc >= M_START) & (hc < M_END)
        va_all = hc >= M_START + (1 - a.holdout) * (M_END - M_START)
        # order M rows as [train..., holdout...] so both are contiguous views (no copies)
        idx = np.flatnonzero(m)
        idx = idx[np.argsort(va_all[idx], kind="stable")]
        cheap = cheap_features(c, s1, pool, rows=idx)
        cm, y = c.iloc[idx].reset_index(drop=True), y_all[idx]
        ntr = int((~va_all[idx]).sum())
        del y_all, c, hc, va_all, m
        gc.collect()
        log(f"features on {len(cm):,} model pairs")
        S = string_features(cm, s1, pool, tables, a.jobs)
        F = list(S.columns) + list(cheap.columns)
        X = np.empty((len(cm), len(F)), np.float32)
        X[:, :S.shape[1]] = S.values
        X[:, S.shape[1]:] = cheap.values
        del S, cheap
        gc.collect()
        log(f"fit lgb: train {ntr:,}  holdout {len(cm) - ntr:,}  pos rate {y.mean():.3f}  feats {len(F)}")
        model = train_lgb(X[:ntr], y[:ntr], rounds=a.rounds, valid=(X[ntr:], y[ntr:]),
                          params=dict(learning_rate=a.lr), feature_name=F)
        va = np.zeros(len(cm), bool)
        va[ntr:] = True
        p = model.predict(X[ntr:], num_threads=0)
        hs1 = np.unique(cm.s1_idx.values[va])
        ids, pids = s1.entity_id.values, pool.entity_id.values
        true = gtp[gtp.s1_id.isin(set(ids[hs1]))].rename(columns={"s1_id": "s1", "pool_id": "pool"})
        pd.DataFrame({"s1_idx": cm.s1_idx.values[va], "pool_idx": cm.pool_idx.values[va], "y": y[va], "p": p}) \
            .to_parquet(W("holdout_preds.parquet"))
        best = (-1, 0.5)
        for thr in np.arange(0.30, 0.96, 0.025):
            sel = select(cm.s1_idx.values[va], cm.pool_idx.values[va], p, thr)
            f, info = f05_macro(pd.DataFrame({"s1": ids[sel.s1.values], "pool": pids[sel.pool.values]}), true, ids[hs1])
            log(f"  thr {thr:.3f}  F0.5 {f:.4f}  {info}")
            best = max(best, (f, thr))
        log(f"HOLDOUT F0.5 {best[0]:.4f} at threshold {best[1]:.3f}")
        imp = pd.Series(model.feature_importance("gain"), F).sort_values(ascending=False)
        log("top features", imp.head(20).round(0).to_dict())
        with open(W("model.pkl"), "wb") as f:
            pickle.dump(dict(model=model, feats=F, thr=float(best[1]), holdout_f05=best[0],
                             noise=(nadd, ndrop, aadd, adrop)), f)
        json.dump(dict(holdout_f05=best[0], threshold=float(best[1]), best_iter=model.best_iteration,
                       features=F, args=vars(a)), open(W("train_report.json"), "w"), indent=1)
        del X, cm, s1, pool
        gc.collect()

    if a.stage == "train":
        log("stage train done")
        return

    # ================================================================ TEST
    s1, pool = load_split(a.data, "test", a.work, a.jobs)
    log("test loaded", len(s1), len(pool), s1.country.value_counts().to_dict())
    if not os.path.exists(W("test_cands.parquet")):
        pr = lgb.Booster(model_file=W("prerank.txt"))
        ix = KeyIndex(s1, pool, a.cap, a.df_scale, a.jobs, cap_composite=a.cap_composite)
        c = generate_candidates(s1, pool, index=ix, prerank=make_pruner(pr, a.k_keep), **wide)
        del ix
        gc.collect()
        c.to_parquet(W("test_cands.parquet"))
    c = pd.read_parquet(W("test_cands.parquet"))
    log(f"test cands {len(c):,}  per S1 {len(c) / len(s1):.1f}")
    if a.stage == "test-cands":
        log("stage test-cands done")
        return
    with open(W("model.pkl"), "rb") as f:
        M = pickle.load(f)
    thr = a.threshold if a.threshold is not None else M["thr"]
    log(f"using threshold {thr:.3f}")
    write_lists(os.path.join(a.out, "candidate_pairs.tsv"), s1.entity_id.values, c.s1_idx.values,
                pool.entity_id.array, c.pool_idx.values, "candidate_entity_ids")
    log("wrote candidate_pairs.tsv")
    cheap = cheap_features(c, s1, pool)
    idf_n = build_idf([s1.nsk, pool.nsk])
    idf_a = build_idf([s1.ask, pool.ask])
    fq = (frequent_tokens([s1.rn, pool.rn]), frequent_tokens([s1.ra, pool.ra]))
    log(f"frequent tokens (fallback gate): name {len(fq[0]):,} addr {len(fq[1]):,}")
    tables = (*idf_n, *idf_a, *M["noise"], *fq)
    p = np.zeros(len(c), np.float32)
    ch = cheap.values
    cheap_cols = list(cheap.columns)
    del cheap
    for k in range(0, len(c), a.predict_chunk):
        sl = slice(k, k + a.predict_chunk)
        S = string_features(c.iloc[sl], s1, pool, tables, a.jobs)
        X = np.hstack([S.values, ch[sl]])
        assert list(S.columns) + cheap_cols == M["feats"], "feature order mismatch"
        p[sl] = M["model"].predict(X, num_threads=0)
        del S, X
        log(f"  predicted {min(k + a.predict_chunk, len(c)):,}/{len(c):,}")
    pd.DataFrame({"s1_idx": c.s1_idx.values, "pool_idx": c.pool_idx.values, "p": p}).to_parquet(W("test_scores.parquet"))
    sel = select(c.s1_idx.values, c.pool_idx.values, p, thr)
    n = write_lists(os.path.join(a.out, "matching_results.tsv"), s1.entity_id.values, sel.s1.values,
                    pool.entity_id.array, sel.pool.values, "matched_entity_ids")
    log(f"DONE: {len(s1):,} S1 rows, {n.mean():.3f} with >=1 match, {len(sel):,} matched pairs "
        f"({len(sel) / len(s1):.2f} per S1)")
    log("by country (share with match):", pd.Series(n).groupby(np.asarray(s1.country)).mean().round(3).to_dict())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/select.py
"""Per-S1 set selection maximizing expected F0.5.

F0.5 for one S1 with k predictions, TP hits and T true matches: 1.25*TP / (k + 0.25*T)
(1 if k = T = 0). With calibrated candidate probabilities q (after the one-to-one rule),
E[F] for predicting the top-k is approximated by 1.25*E[TP_k] / (k + 0.25*E[T]),
and the empty set scores P(T = 0) = prod(1 - q). We pick the best k per S1.
"""
import numpy as np
import pandas as pd


def one_to_one(s1_idx, pool_idx, p):
    df = pd.DataFrame({"s1": s1_idx, "pool": pool_idx, "p": p})
    best = df.groupby("pool").p.transform("max")
    return df[df.p >= best].drop_duplicates("pool")


def expected_f_select(s1_idx, pool_idx, p, miss_prior=0.0, min_p=0.0, power=1.0):
    """miss_prior: expected number of true matches not among candidates (blocking misses) per S1."""
    df = one_to_one(s1_idx, pool_idx, p)
    q_all = df.p.values ** power
    df = df.assign(q=q_all).sort_values(["s1", "q"], ascending=[True, False])
    g = df.s1.values
    q = df.q.values
    first = np.r_[True, g[1:] != g[:-1]]
    starts = np.flatnonzero(first)
    grp = np.cumsum(first) - 1
    k = np.arange(len(g)) - starts[grp] + 1
    csum = np.cumsum(q)
    etp = csum - np.r_[0, csum][starts][grp]
    et = np.add.reduceat(q, starts)[grp] + miss_prior
    ef = 1.25 * etp / (k + 0.25 * et)
    logq0 = np.add.reduceat(np.log1p(-np.clip(q, 0, 1 - 1e-7)), starts)
    p_empty = np.exp(logq0) * np.exp(-miss_prior)
    # best k per group
    best_ef = pd.Series(ef).groupby(grp).transform("max").values
    kbest = pd.Series(np.where(ef >= best_ef, k, 10 ** 9)).groupby(grp).transform("min").values
    choose = (k <= kbest) & (best_ef[...] > p_empty[grp]) & (q >= min_p)
    return df[choose][["s1", "pool", "p"]]

In [ ]:
%%writefile src/siblings.py
"""Stage-2 collective features.

A true S2/S3 record usually has 1-5 sibling records (same S1) that are near-identical to it.
Given stage-1 probabilities p1, compare every candidate with the S1's other top candidates:
if it is a near-duplicate of a confident sibling it is almost surely a match, even when its
own name/number was perturbed. Also adds p1 competition context.
"""
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.process import cpdist

from .features import context_features

TOPS = 3


def _topk_per_group(g, v, k):
    order = np.lexsort((-v, g))
    gs = g[order]
    first = np.r_[True, gs[1:] != gs[:-1]]
    start = np.flatnonzero(first)
    rank = np.arange(len(g)) - np.repeat(start, np.diff(np.r_[start, len(g)]))
    keep = rank < k
    return order[keep], gs[keep], rank[keep]


def sibling_features(c, p1, pool, workers=-1):
    """c: DataFrame [s1_idx, pool_idx] (row-aligned with p1). pool: normalized pool frame."""
    n = len(c)
    s1 = c.s1_idx.values
    pj = c.pool_idx.values
    # top-(TOPS+1) candidates per S1 by p1 (one extra so we can exclude the pair itself)
    rows, gs, rk = _topk_per_group(s1, p1, TOPS + 1)
    top = pd.DataFrame({"s1_idx": gs, "sib_row": rows, "rk": rk})
    pairs = pd.DataFrame({"s1_idx": s1, "row": np.arange(n)}).merge(top, on="s1_idx")
    pairs = pairs[pairs.row != pairs.sib_row]
    pairs = pairs[pairs.groupby("row").cumcount() < TOPS]
    r = pairs.row.values
    sr = pairs.sib_row.values
    a_pool, b_pool = pj[r], pj[sr]
    nm, ad, nu = pool.n_clean.values, pool.a_clean.values, pool.nums.values
    s_name = cpdist(list(nm[a_pool]), list(nm[b_pool]), scorer=fuzz.token_set_ratio, workers=workers, dtype=np.float32)
    s_addr = cpdist(list(ad[a_pool]), list(ad[b_pool]), scorer=fuzz.token_set_ratio, workers=workers, dtype=np.float32)
    s_num = cpdist(list(nu[a_pool]), list(nu[b_pool]), scorer=fuzz.token_set_ratio, workers=workers, dtype=np.float32)
    sp = p1[sr]
    both = (s_name + s_addr) / 2
    df = pd.DataFrame({"row": r, "sn": s_name, "sa": s_addr, "su": s_num, "sb": both, "sp": sp,
                       "wsb": both * sp})
    g = df.groupby("row")
    out = pd.DataFrame(index=np.arange(n))
    agg = g.agg(sib_name_max=("sn", "max"), sib_addr_max=("sa", "max"), sib_num_max=("su", "max"),
                sib_both_max=("sb", "max"), sib_wboth_max=("wsb", "max"), sib_p_max=("sp", "max"),
                sib_p_sum=("sp", "sum"), sib_n=("sp", "size"))
    out = out.join(agg)
    # p1 of the sibling most similar to this record
    idx = df.sb.values.argsort(kind="stable")
    best = df.iloc[idx].drop_duplicates("row", keep="last").set_index("row")
    out["sib_p_of_most_similar"] = best.sp
    out["sib_sim_of_top1"] = df[df.sp >= df.groupby("row").sp.transform("max")].drop_duplicates("row").set_index("row").sb
    out = out.fillna(-1).astype(np.float32)
    ctx = context_features(pd.DataFrame({"s1_idx": s1, "pool_idx": pj, "p1": p1}), "p1", "p1")
    out = pd.concat([out, ctx.reset_index(drop=True)], axis=1)
    out["p1"] = p1.astype(np.float32)
    g1 = pd.Series(p1).groupby(s1)
    out["p1_sum_s1"] = g1.transform("sum").values.astype(np.float32)
    out["p1_n50_s1"] = pd.Series(p1 > 0.5).groupby(s1).transform("sum").values.astype(np.float32)
    return out

In [ ]:
hits = glob.glob("/kaggle/input/**/train_source1.tsv", recursive=True)
assert hits, "Add the challenge dataset as an Input first"
DATA = os.path.dirname(os.path.dirname(hits[0]))   # .../dataset
ROOT = os.path.dirname(DATA)                       # .../student_resource
print(DATA)

In [ ]:
# Attach outputs of earlier jobs as Notebook Output inputs; their cached stages are reused (symlinked).
# resume: T output (train_norm_*, train_cands, prerank.txt) + S output (test_norm_*, test_cands)
# predict: T output (model.pkl, prerank.txt) + S output
imp = sorted({os.path.dirname(p) for pat in ("model.pkl", "prerank.txt", "train_cands.parquet", "train_norm_s1.parquet",
                                             "test_cands.parquet", "test_norm_s1.parquet")
              for p in glob.glob(f"/kaggle/input/**/{pat}", recursive=True)})
print(imp)
EXTRA = " ".join(f'"{d}"' for d in imp)
RUN_STAGE = "all"

In [ ]:
!python -m src.run --data "$DATA" --work /kaggle/working/work --out /kaggle/working/output --stage {RUN_STAGE} --extra-work {EXTRA} 2>&1 | grep -v "Warning\]" | tee /kaggle/working/run.log

In [ ]:
!python "$ROOT/utils/validate_submission.py" --matching /kaggle/working/output/matching_results.tsv \
    --candidate /kaggle/working/output/candidate_pairs.tsv --test-dir "$DATA/test"

In [ ]:
!rm -rf /kaggle/working/ber/src/__pycache__
!du -sh /kaggle/working/*; ls -la /kaggle/working/work /kaggle/working/output 2>/dev/null